# Notebook 6 — Blueprint Assembly & Emission

The previous notebooks covered every building block in isolation — features, classes, influences, and the DAG. This notebook shows how those pieces come together through the `Blueprint` object itself.

You'll see:
- How to construct a `Blueprint` and register components
- The method-chaining pattern for concise assembly
- `validate()` — what it checks and when to call it
- `describe()` — a human-readable summary of your blueprint
- `emit()` — the full generation pipeline, step by step
- `emit(describe=True)` and `emit(manifest=...)`
- The output format methods: `to_csv()`, `to_json()`, `to_manifest()`
- What a manifest file contains and how to use it

In [1]:
from blueprint import Blueprint, Feature, Class, Influence
from blueprint.emitter.formats import to_manifest

import numpy as np
import pandas as pd
import json
import tempfile
import os

## Creating a Blueprint

`Blueprint(n=, seed=)` is the entry point. It takes two parameters:

- **`n`** — the number of rows to generate when `emit()` is called
- **`seed`** — the master random seed; the same seed always produces identical data

Both are required. There are no defaults for `n` (you always state how many rows you want) and `seed` defaults to 42.

In [2]:
bp = Blueprint(n=5, seed=42)
print(bp.n, bp.seed)
print(type(bp))

5 42
<class 'blueprint.core.blueprint.Blueprint'>


A freshly created `Blueprint` has empty feature, class, and influence lists. Nothing is generated until you call `emit()`.

## Registering Components

Three methods register the three building-block types:

| Method | Registers |
|---|---|
| `add_feature(*features)` | `Feature` objects — the columns of the output DataFrame |
| `add_class(*classes)` | `Class` objects — population segments with overridden feature parameters |
| `add_influence(*influences)` | `Influence` objects — causal edges that adjust column values |

All three methods accept one or more arguments of the corresponding type and return `self`, enabling method chaining (shown in the next section).

In [3]:
bp = Blueprint(n=5, seed=42)
bp.add_feature(Feature("age",    dtype=int,   base=35,    std=10,    clip=(18, 80)))
bp.add_feature(Feature("income", dtype=float, base=60000, std=15000, clip=(0, None)))
print("Features:", [f.name for f in bp.features])

Features: ['age', 'income']


In [4]:
bp.add_class(Class("senior", when=("age", ">=", 60)))
print("Classes:", [c.name for c in bp._classes])

Classes: ['senior']


In [5]:
bp.add_influence(Influence("age").on("income", effect="+500 per unit"))
print("Influences:", len(bp._influences))

Influences: 1


### Variadic registration

Each `add_*` method accepts multiple arguments in a single call — no need to call `add_feature` once per feature:

In [6]:
bp2 = Blueprint(n=5, seed=1)
bp2.add_feature(
    Feature("x", dtype=float, base=0, std=1),
    Feature("y", dtype=float, base=0, std=1),
    Feature("z", dtype=float, base=0, std=1),
)
print("All three at once:", [f.name for f in bp2.features])

All three at once: ['x', 'y', 'z']


## Method Chaining

Because every `add_*` method returns `self`, you can chain all registration calls onto the constructor in one expression. This is the preferred pattern for building blueprints:

In [7]:
bp = (
    Blueprint(n=1000, seed=7)
    .add_feature(
        Feature("age",    dtype=int,   base=35,    std=10,    clip=(18, 80)),
        Feature("income", dtype=float, base=60000, std=15000, clip=(0, None), derived=True),
    )
    .add_class(Class("senior", when=("age", ">=", 60)))
    .add_influence(Influence("age").on("income", effect="+500 per unit"))
)

df = bp.emit()
print(df.head())
print(f"\nshape: {df.shape}")

   age   income
0   35  17500.0
1   38  19000.0
2   32  16000.0
3   26  13000.0
4   30  15000.0

shape: (1000, 2)


The chained form reads like a description of the dataset: 1000 rows, seeded with 7, two features, one class, one influence. Everything is defined in one place and the blueprint is fully assembled before `emit()` is called.

## `validate()` — Checking Your Blueprint

`validate()` performs a dry-run consistency check on the blueprint without generating any data. It raises `ValueError` if it finds:

- A class `when=` condition that references a feature name that doesn't exist
- A class `override()` that references a feature name that doesn't exist
- An `Influence` source or target that isn't a registered feature
- A cycle in the influence graph (`BlueprintCycleError`)

Call `validate()` after assembly, before `emit()`, to catch wiring errors early — especially in programmatically built blueprints where typos are easy to introduce.

In [8]:
bp.validate()
print("validate() passed — no exceptions raised")

validate() passed — no exceptions raised


In [9]:
# Undefined influence target
try:
    bad = Blueprint(n=5, seed=1)
    bad.add_feature(Feature("x", dtype=float, base=0, std=1))
    bad.add_influence(Influence("x").on("y_undefined", effect="+1"))
    bad.validate()
except ValueError as e:
    print(f"Undefined target    → ValueError: {e}")

# Class that references a non-existent column
try:
    bad2 = Blueprint(n=5, seed=1)
    bad2.add_feature(Feature("score", dtype=float, base=50, std=10))
    bad2.add_class(Class("power", when=("missing_col", ">=", 90)))
    bad2.validate()
except ValueError as e:
    print(f"Undefined class col → ValueError: {e}")

Undefined target    → ValueError: Influence: target 'y_undefined' is not a defined feature
Undefined class col → ValueError: Class 'power': 'when' references undefined feature 'missing_col'


`validate()` is silent on success — it returns `None` and raises nothing. The pattern is to call it right before `emit()` (or even before `describe()`) whenever you've assembled a blueprint programmatically.

## `describe()` — Human-Readable Summary

`describe()` prints a compact, human-readable summary of everything registered in the blueprint. It shows:

- `n` and `seed`
- Every feature with its dtype and distribution parameters
- Every class with its condition and any overrides
- Every influence edge and its effect string
- The topological evaluation order (if influences or computed features are present)

In [10]:
bp.describe()

Blueprint(n=1000, seed=7)

Features (2):
  age                  <class 'int'> base=35, std=10, clip=(18, 80)
  income               <class 'float'> base=60000, std=15000, clip=(0, None)

Classes (1):
  senior               when: age >= 60

Influences (1):
  age → income  +500 per unit

Evaluation order: ['age', 'income']


The **evaluation order** line shows the topologically sorted list of feature names — the exact sequence in which Blueprint will process influences during `emit()`. Features with no dependencies appear early; features that depend on upstream columns appear after their sources.

`describe()` runs the same topological sort as `emit()` does internally, so if you see the right order here, the data will be generated correctly.

## `emit()` — Full Pipeline Walkthrough

Calling `emit()` runs the entire generation pipeline and returns a `pandas.DataFrame` with one column per registered feature and `n` rows.

The pipeline proceeds in this order:

1. **Seed the RNG** — a `numpy.random.default_rng(seed)` is created for shared use; features with an explicit `seed=` get their own RNG
2. **Generate base values** — each non-computed feature calls `.generate()` and `.apply_modifiers()`; computed features get a zeros placeholder
3. **Build class masks** — each class resolves which rows belong to it using its `when=` condition
4. **Apply class overrides** — rows matching a class get regenerated values for any overridden features
5. **Apply influences** — influence edges are applied in topological order, updating target columns based on source values
6. **Evaluate computed columns** — `dtype="computed"` features run their `formula=` lambda against the fully-updated DataFrame
7. **Return the DataFrame**

Here's the blueprint we'll use for the remaining examples — a simplified real-estate dataset:

In [11]:
bp_re = (
    Blueprint(n=6, seed=42)
    .add_feature(
        Feature("sqft",  dtype=int,   base=1800,   std=400,   clip=(500, 5000)),
        Feature("price", dtype=float, base=300000,  std=80000, clip=(0, None), derived=True),
        Feature("tax",   dtype=float, base=0,       std=0,     derived=True),
    )
    .add_class(Class("luxury", when=("sqft", ">=", 2500)))
    .add_influence(
        Influence("sqft").on("price", effect="+155 per unit"),
        Influence("price").on("tax",  effect="+0.012 per unit"),
    )
)

df = bp_re.emit()
print(df.round(2))
print()
print("tax ≈ price × 0.012:", np.allclose(df.tax, df.price * 0.012, rtol=1e-5))

   sqft     price      tax
0  1922  297910.0  3574.92
1  1384  214520.0  2574.24
2  2100  325500.0  3906.00
3  2176  337280.0  4047.36
4  1020  158100.0  1897.20
5  1279  198245.0  2378.94

tax ≈ price × 0.012: True


The assertion confirms end-to-end correctness: `sqft` drove `price`, then `price` drove `tax`, in the right order.

## `emit(describe=True)`

Passing `describe=True` to `emit()` calls `describe()` on the blueprint before generating data. This is a convenience flag for interactive use — you get the summary and the DataFrame in one call without having to call both methods separately.

In [12]:
df = bp_re.emit(describe=True)

Blueprint(n=6, seed=42)

Features (3):
  sqft                 <class 'int'> base=1800, std=400, clip=(500, 5000)
  price                <class 'float'> base=300000, std=80000, clip=(0, None)
  tax                  <class 'float'> base=0, std=0

Classes (1):
  luxury               when: sqft >= 2500

Influences (2):
  sqft → price  +155 per unit
  price → tax  +0.012 per unit

Evaluation order: ['sqft', 'price', 'tax']


The description is printed before generation begins; the returned `df` is identical to a plain `emit()` call.

## `emit(manifest=...)`

Passing a file path to `manifest=` writes a JSON sidecar file alongside the dataset. The manifest records the exact configuration used to generate the data:

- `seed` and `n_rows`
- Feature names and dtypes
- Class names and their `when=` conditions (as strings)
- The influence graph: a dict mapping each source feature to its target features

Manifests are useful for reproducibility audits — you can look at a CSV file months later and know exactly how it was generated.

In [13]:
with tempfile.TemporaryDirectory() as tmpdir:
    manifest_path = os.path.join(tmpdir, "manifest.json")
    df = bp_re.emit(manifest=manifest_path)
    with open(manifest_path) as f:
        manifest = json.load(f)

print(json.dumps(manifest, indent=2))

{
  "seed": 42,
  "n_rows": 6,
  "features": {
    "sqft": "<class 'int'>",
    "price": "<class 'float'>",
    "tax": "<class 'float'>"
  },
  "classes": {
    "luxury": "('sqft', '>=', 2500)"
  },
  "influence_graph": {
    "sqft": [
      "price"
    ],
    "price": [
      "tax"
    ]
  }
}


## Output Formats: `to_csv()` and `to_json()`

Blueprint provides two convenience methods that call `emit()` internally and then write the result to disk:

| Method | Writes | Returns |
|---|---|---|
| `to_csv(path, index=False, encoding="utf-8")` | CSV file | the emitted DataFrame |
| `to_json(path, orient="records", indent=2, date_format="iso", manifest=None)` | JSON file | the emitted DataFrame |

Both methods return the DataFrame so you can continue working with the data in-memory after writing.

In [14]:
with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = os.path.join(tmpdir, "realestate.csv")
    df_out = bp_re.to_csv(csv_path)
    print(f"Rows written: {len(df_out)}")
    with open(csv_path) as f:
        lines = f.readlines()[:4]
    print("".join(lines), end="")

Rows written: 6
sqft,price,tax
1922,297910.0,3574.92
1384,214520.0,2574.2400000000002
2100,325500.0,3906.0


In [15]:
with tempfile.TemporaryDirectory() as tmpdir:
    json_path = os.path.join(tmpdir, "realestate.json")
    df_out = bp_re.to_json(json_path)
    with open(json_path) as f:
        print(f.read())

[
  {
    "sqft":1922,
    "price":297910.0,
    "tax":3574.92
  },
  {
    "sqft":1384,
    "price":214520.0,
    "tax":2574.24
  },
  {
    "sqft":2100,
    "price":325500.0,
    "tax":3906.0
  },
  {
    "sqft":2176,
    "price":337280.0,
    "tax":4047.36
  },
  {
    "sqft":1020,
    "price":158100.0,
    "tax":1897.2
  },
  {
    "sqft":1279,
    "price":198245.0,
    "tax":2378.94
  }
]


### Combining `to_json` with a manifest

`to_json` accepts an optional `manifest=` path. When provided, Blueprint writes the JSON data file and the manifest sidecar in one call — the same manifest format as `emit(manifest=...)`:

In [16]:
with tempfile.TemporaryDirectory() as tmpdir:
    json_path     = os.path.join(tmpdir, "realestate.json")
    manifest_path = os.path.join(tmpdir, "manifest.json")
    df_out = bp_re.to_json(json_path, manifest=manifest_path)
    with open(manifest_path) as f:
        print(json.dumps(json.load(f), indent=2))

{
  "seed": 42,
  "n_rows": 6,
  "features": {
    "sqft": "<class 'int'>",
    "price": "<class 'float'>",
    "tax": "<class 'float'>"
  },
  "classes": {
    "luxury": "('sqft', '>=', 2500)"
  },
  "influence_graph": {
    "sqft": [
      "price"
    ],
    "price": [
      "tax"
    ]
  }
}


## Manifest Files

A manifest is a JSON file that records the complete configuration needed to understand — and reproduce — a generated dataset. It contains five top-level keys:

| Key | Type | Description |
|---|---|---|
| `seed` | int | The master random seed |
| `n_rows` | int | Number of rows in the dataset |
| `features` | dict | `{name: dtype_string}` for every feature |
| `classes` | dict | `{name: when_condition_string}` for every class |
| `influence_graph` | dict | `{source: [target, ...]}` adjacency list of the dependency DAG |

You can also call `to_manifest()` directly from `blueprint.emitter.formats` to write a manifest without calling `emit()`. This is useful when you want to checkpoint the blueprint definition at design time, before any data is generated:

In [17]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "my_manifest.json")
    to_manifest(
        path,
        seed=42,
        n_rows=1000,
        features={"sqft": "int", "price": "float", "tax": "float"},
        classes={"luxury": "sqft >= 2500"},
        influence_graph={"sqft": ["price"], "price": ["tax"]},
    )
    with open(path) as f:
        print(json.dumps(json.load(f), indent=2))

{
  "seed": 42,
  "n_rows": 1000,
  "features": {
    "sqft": "int",
    "price": "float",
    "tax": "float"
  },
  "classes": {
    "luxury": "sqft >= 2500"
  },
  "influence_graph": {
    "sqft": [
      "price"
    ],
    "price": [
      "tax"
    ]
  }
}


### When to use manifests

Manifests are lightweight but powerful for reproducibility workflows:

- **Training data provenance** — store a manifest alongside every train/test CSV so you can always reconstruct the exact dataset
- **Dataset versioning** — the manifest is a diff-friendly, human-readable record of what changed between dataset versions
- **Automated re-generation** — a downstream script can read a manifest and rebuild the dataset from scratch by calling `Blueprint(n=manifest["n_rows"], seed=manifest["seed"])` and wiring up the same structure

---

## Summary

| Method | What it does |
|---|---|
| `Blueprint(n, seed)` | Creates a blueprint with row count and seed |
| `.add_feature(*features)` | Registers Feature objects; returns `self` |
| `.add_class(*classes)` | Registers Class objects; returns `self` |
| `.add_influence(*influences)` | Registers Influence objects; returns `self` |
| `.validate()` | Checks references and cycle-freedom; raises on error |
| `.describe()` | Prints a human-readable blueprint summary |
| `.emit()` | Runs the full generation pipeline; returns a DataFrame |
| `.emit(describe=True)` | Calls `describe()` then `emit()` |
| `.emit(manifest=path)` | Emits and writes a JSON manifest sidecar |
| `.to_csv(path)` | Emits and writes a CSV file; returns DataFrame |
| `.to_json(path, manifest=path)` | Emits and writes a JSON file (+ optional manifest); returns DataFrame |